In [9]:
import numpy as np
import pandas as pd
import matplotlib as plt


In [10]:
df = pd.read_csv("../data/used_cars.csv")

In [11]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


CLEANING DATA

In [12]:
df.drop(['engine', 'ext_col', 'int_col', 'model'], axis=1, inplace=True)

In [13]:
df['price'] = df['price'].str.replace('$', '').str.replace(',', '').astype(int)
df['milage'] = df['milage'].str.replace('mi.', '').str.replace(',', '').astype(int)
df[['milage', 'price']].head()

df['fuel_type'] = df['fuel_type'].str.replace('–', 'Gasoline')

In [14]:
# Removing outliers
df = df[df["price"] < 300000] # REMOVING OUTLIERS

In [15]:
df['fuel_type'] = df['fuel_type'].fillna('Gasoline')
df['accident'] = df['accident'].fillna('None reported')

In [16]:
# df.sample(8)
df['clean_title'] = df['clean_title'].fillna('Unknown')
df['clean_title'].unique()

<StringArray>
['Yes', 'Unknown']
Length: 2, dtype: str

In [17]:
df.isnull().sum()

brand           0
model_year      0
milage          0
fuel_type       0
transmission    0
accident        0
clean_title     0
price           0
dtype: int64

In [18]:
df['fuel_type'].unique()

<StringArray>
[ 'E85 Flex Fuel',       'Gasoline',         'Hybrid',         'Diesel',
 'Plug-In Hybrid',  'not supported']
Length: 6, dtype: str

FIXING WIDE DATA

In [19]:
cols_to_encode = ['fuel_type', 'accident', 'clean_title']
df = pd.get_dummies(data=df, columns=cols_to_encode, drop_first=True, dtype=int)
df.head()

,brand,model_year,milage,transmission,price,fuel_type_E85 Flex Fuel,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid,fuel_type_not supported,accident_None reported,clean_title_Yes
0,Ford,2013,51000,6-Speed A/T,10300,1,0,0,0,0,0,1
1,Hyundai,2021,34742,8-Speed Automatic,38005,0,1,0,0,0,0,1
2,Lexus,2022,22372,Automatic,54598,0,1,0,0,0,1,0
3,INFINITI,2015,88900,7-Speed A/T,15500,0,0,1,0,0,1,1
4,Audi,2021,9835,8-Speed Automatic,34999,0,1,0,0,0,1,0


In [20]:
def simplify_transmission(x):
    x = str(x).lower()
    
    if "manual" in x or "m/t" in x:
        return "Manual"
    elif "cvt" in x:
        return "CVT"
    elif "dual" in x or "dct" in x:
        return "Dual-Clutch"
    elif "auto" in x or "a/t" in x:
        return "Automatic"
    else:
        return "Other"

df["transmission"] = df["transmission"].apply(simplify_transmission)
df = pd.get_dummies(df, columns=['transmission'], drop_first=True, dtype=int)

In [21]:
df['brand'].value_counts()

brand
Ford             385
BMW              375
Mercedes-Benz    314
Chevrolet        292
Audi             200
Toyota           199
Porsche          198
Lexus            163
Jeep             143
Land             130
Nissan           116
Cadillac         106
GMC               91
RAM               91
Dodge             89
Tesla             87
Kia               76
Hyundai           72
Acura             64
Subaru            64
Mazda             64
Honda             63
INFINITI          59
Volkswagen        59
Lincoln           52
Jaguar            47
Volvo             38
MINI              33
Maserati          33
Bentley           32
Buick             30
Chrysler          28
Genesis           20
Mitsubishi        20
Alfa              19
Lamborghini       19
Rivian            17
Hummer            16
Pontiac           15
Aston              9
Ferrari            9
Scion              6
Saturn             5
McLaren            5
FIAT               5
Lotus              4
Rolls-Royce        4
Lucid  

In [22]:
threshold = 30
brand_counts = df['brand'].value_counts()
valid_cars = brand_counts[brand_counts >= threshold].index

df['brand'] = df['brand'].apply(
    lambda x: x if x in valid_cars else 'Other'
)

In [23]:
df = pd.get_dummies(df, columns=['brand'], drop_first=True, dtype=int)
df

,model_year,milage,price,fuel_type_E85 Flex Fuel,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid,fuel_type_not supported,accident_None reported,clean_title_Yes,...,brand_Mercedes-Benz,brand_Nissan,brand_Other,brand_Porsche,brand_RAM,brand_Subaru,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo
0,2013,51000,10300,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2021,34742,38005,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,2022,22372,54598,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2015,88900,15500,0,0,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,2021,9835,34999,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,2018,53705,25900,0,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
4005,2022,10900,53900,0,1,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4006,2022,2116,90998,0,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4007,2020,33000,62999,0,1,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0


CREATING MODEL

In [24]:
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [25]:
X = df.drop('price', axis=1)
y = np.log(df['price'])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [27]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


TESTING`

In [28]:
# EVALUATION
y_pred = regressor.predict(X_test)

R2 = r2_score(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
print(R2, RMSE)

0.6823026586134358 0.4816609466733941


In [29]:
import pickle
pickle.dump(regressor, open('../models/car_price_model.pkl', 'wb'))

In [30]:
pickle.dump(X.columns, open('../models/model_columns.pkl', 'wb'))